<a href="https://colab.research.google.com/github/RafaelaMlucca/AnaliseViolMulher/blob/main/01_ingestao_dados_sinan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline SINAN-VIOL — Pré-processamento

**Projeto:** Violência contra a mulher  
**Autora:** Rafaela Lucca


## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install pysus --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [9]:
import gc
import glob
from pathlib import Path
import pandas as pd

DRIVE = Path('/content/drive/MyDrive/projeto_violencia_mulher')
PARTES = DRIVE / 'partes'  # parquets parciais (1 por ano)
DRIVE.mkdir(parents=True, exist_ok=True)
PARTES.mkdir(parents=True, exist_ok=True)
print(f'Salvando em: {DRIVE}')

Salvando em: /content/drive/MyDrive/projeto_violencia_mulher


In [13]:
disponiveis = pd.read_parquet(paths[0]).columns.tolist()
print(disponiveis)

['TP_NOT', 'ID_AGRAVO', 'DT_NOTIFIC', 'SEM_NOT', 'NU_ANO', 'SG_UF_NOT', 'ID_MUNICIP', 'ID_UNIDADE', 'DT_OCOR', 'SEM_PRI', 'ANO_NASC', 'NU_IDADE_N', 'CS_SEXO', 'CS_GESTANT', 'CS_RACA', 'CS_ESCOL_N', 'SG_UF', 'ID_MN_RESI', 'ID_PAIS', 'NDUPLIC', 'DT_INVEST', 'ID_OCUPA_N', 'SIT_CONJUG', 'DEF_TRANS', 'DEF_FISICA', 'DEF_MENTAL', 'DEF_VISUAL', 'DEF_AUDITI', 'TRAN_MENT', 'TRAN_COMP', 'DEF_OUT', 'DEF_ESPEC', 'SG_UF_OCOR', 'ID_MN_OCOR', 'HORA_OCOR', 'LOCAL_OCOR', 'LOCAL_ESPE', 'OUT_VEZES', 'LES_AUTOP', 'VIOL_FISIC', 'VIOL_PSICO', 'VIOL_TORT', 'VIOL_SEXU', 'VIOL_TRAF', 'VIOL_FINAN', 'VIOL_NEGLI', 'VIOL_INFAN', 'VIOL_LEGAL', 'VIOL_OUTR', 'VIOL_ESPEC', 'AG_FORCA', 'AG_ENFOR', 'AG_OBJETO', 'AG_CORTE', 'AG_QUENTE', 'AG_ENVEN', 'AG_FOGO', 'AG_AMEACA', 'AG_OUTROS', 'AG_ESPEC', 'SEX_ASSEDI', 'SEX_ESTUPR', 'SEX_PUDOR', 'SEX_PORNO', 'SEX_EXPLO', 'SEX_OUTRO', 'SEX_ESPEC', 'PEN_ORAL', 'PEN_ANAL', 'PEN_VAGINA', 'PROC_DST', 'PROC_HIV', 'PROC_HEPB', 'PROC_SANG', 'PROC_SEMEN', 'PROC_VAGIN', 'PROC_CONTR', 'PROC_

In [14]:
COLS = [
    # Identificação
    'NU_ANO', 'SG_UF_NOT', 'TPUNINOT', 'DT_NOTIFIC', 'DT_OCOR',

    # Vítima
    'CS_SEXO', 'NU_IDADE_N', 'CS_RACA', 'CS_ESCOL_N', 'CS_GESTANT',
    'ORIENT_SEX', 'IDENT_GEN', 'SIT_CONJUG',
    'DEF_TRANS', 'DEF_FISICA', 'DEF_MENTAL',

    # Ocorrência
    'LOCAL_OCOR', 'HORA_OCOR', 'OUT_VEZES', 'LES_AUTOP', 'CICL_VID',

    # Tipos de violência (3 alvos + outros para EDA)
    'VIOL_FISIC', 'VIOL_PSICO', 'VIOL_TORT', 'VIOL_SEXU',
    'VIOL_TRAF', 'VIOL_FINAN', 'VIOL_NEGLI', 'VIOL_INFAN',
    'VIOL_LEGAL', 'VIOL_OUTR',

    # Meios de agressão
    'AG_FORCA', 'AG_ENFOR', 'AG_OBJETO', 'AG_CORTE',
    'AG_QUENTE', 'AG_ENVEN', 'AG_FOGO', 'AG_AMEACA', 'AG_OUTROS',

    # Autor
    'AUTOR_SEXO', 'AUTOR_ALCO',

    # Relação com agressor
    'REL_PAI', 'REL_MAE', 'REL_CONJ', 'REL_EXCON', 'REL_NAMO',
    'REL_EXNAM', 'REL_FILHO', 'REL_IRMAO',
    'REL_CONHEC', 'REL_DESCO', 'REL_CUIDA', 'REL_PATRAO',
]

## 2. Baixar dados do SINAN

Últimos 5 anos (2020-2024). O `pysus` cacheia em `/root/pysus/`.

In [15]:
from pysus import SINAN

ANOS = list(range(2020, 2025))

sinan = SINAN().load()
files = sinan.get_files(dis_code='VIOL', year=ANOS)
sinan.download(files)

paths = sorted(glob.glob('/root/pysus/VIOLBR*.parquet'))
paths = [p for p in paths if any(f'VIOLBR{a%100:02d}' in p for a in ANOS)]
print(f'{len(paths)} arquivos disponíveis')

49738833it [00:00, 12364119374.58it/s]

5 arquivos disponíveis


## 3. Processar arquivo por arquivo

Para cada parquet:
1. Lê
2. Filtra apenas mulheres (`CS_SEXO == 'F'`)
3. Guarda só o filtrado em uma lista
4. Libera o resto da memória

Isso evita ter os 10 anos completos em RAM ao mesmo tempo.

In [16]:
for p in paths:
    nome = Path(p).stem  # ex: VIOLBR15
    saida = PARTES / f'{nome}_mulher.parquet'

    if saida.exists():
        print(f'{nome}: já existe, pulando')
        continue

    # Lê só as colunas que importam
    df_ano = pd.read_parquet(p, columns=COLS)
    n_total = len(df_ano)

    # Filtra mulheres (sem .copy() — usa o próprio resultado da indexação)
    df_ano = df_ano[df_ano['CS_SEXO'] == 'F'].reset_index(drop=True)
    n_mulher = len(df_ano)

    # Salva direto no Drive
    df_ano.to_parquet(saida, index=False)
    print(f'{nome}: {n_total:>7,} -> {n_mulher:>7,} mulheres')

    # Libera memória
    del df_ano
    gc.collect()

print('\nProcessamento concluído.')

VIOLBR20: 326,503 -> 233,072 mulheres
VIOLBR21: 373,262 -> 268,884 mulheres
VIOLBR22: 442,680 -> 318,137 mulheres
VIOLBR23: 588,388 -> 419,967 mulheres
VIOLBR24: 616,548 -> 437,828 mulheres

Processamento concluído.


In [17]:
# Lê os parquets parciais do Drive e concatena
partes_paths = sorted(PARTES.glob('*_mulher.parquet'))
print(f'Concatenando {len(partes_paths)} partes...')

dfs = [pd.read_parquet(p) for p in partes_paths]
mulher = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

print(f'Base de mulheres: {mulher.shape}')

Concatenando 5 partes...
Base de mulheres: (1677888, 54)


In [18]:
# Cria os 3 alvos: 1=Sim, 0=Não, NaN=Ignorado
for col in ['VIOL_FISIC', 'VIOL_PSICO', 'VIOL_SEXU']:
    novo = 'y_' + col.replace('VIOL_', '').lower()
    mulher[novo] = mulher[col].map({'1': 1, '2': 0})
    print(f'{novo}: {mulher[novo].value_counts(dropna=False).to_dict()}')

y_fisic: {0.0: 829018, 1.0: 827465, nan: 21405}
y_psico: {0.0: 1253661, 1.0: 384248, nan: 39979}
y_sexu: {0.0: 1365662, 1.0: 271400, nan: 40826}


## 4. Salvar bronze (base bruta) no Drive

Snapshot da base como veio. Se algo der errado depois, voltamos aqui sem precisar baixar de novo.

In [19]:
mulher.to_parquet(DRIVE / 'viol_mulher.parquet', index=False)
print(f'Salvo: {DRIVE / "viol_mulher.parquet"}')
print(f'Shape final: {mulher.shape}')

Salvo: /content/drive/MyDrive/projeto_violencia_mulher/viol_mulher.parquet
Shape final: (1677888, 57)


In [20]:
df = mulher

## 5. Primeira olhada nos dados

In [11]:
df.head()

,TP_NOT,ID_AGRAVO,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_UNIDADE,DT_OCOR,SEM_PRI,...,MPU,DELEG_CRIA,DELEG_MULH,DELEG,INFAN_JUV,DEFEN_PUBL,DT_ENCERRA,y_fisic,y_psico,y_sexu
0,2,Y09,20150624,201525,2015,15,150680,3736695,20150624,201525,...,1,2,2,2,2,2,20160624,1.0,1.0,0.0
1,2,Y09,20150702,201526,2015,15,150680,2330121,20150702,201525,...,2,2,2,2,2,2,20160702,0.0,1.0,0.0
2,2,Y09,20150704,201526,2015,33,330240,2276542,20150704,201526,...,2,2,2,2,2,2,20160704,1.0,0.0,0.0
3,2,Y09,20150708,201527,2015,26,261370,6507557,20150708,201527,...,2,2,2,2,2,2,20150708,1.0,0.0,0.0
4,2,Y09,20150709,201527,2015,31,311250,2194260,20150709,201527,...,2,2,2,1,2,2,20150709,1.0,1.0,0.0


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2775742 entries, 0 to 2775741
Columns: 163 entries, TP_NOT to y_sexu
dtypes: float64(3), object(160)
memory usage: 3.4+ GB


In [24]:
# Quantos registros por ano?
print(mulher['NU_ANO'].value_counts().sort_index())

NU_ANO
        233072
2021    268884
2022    318137
2023    419967
2024    437828
Name: count, dtype: int64


## 8. Salvar base pronta para EDA

In [23]:
mulher.to_parquet(DRIVE / 'viol_mulher.parquet', index=False)
print(f'Salvo: {DRIVE / "viol_mulher.parquet"}')
print(f'Shape: {mulher.shape}')

Salvo: /content/drive/MyDrive/projeto_violencia_mulher/viol_mulher.parquet
Shape: (1677888, 57)
